# Smart Blackjack - Entraînement du modèle de cartes

Ce notebook entraîne un **YOLOv8n** à reconnaître les 52 cartes, puis l'exporte
au format **NCNN** pour le Raspberry Pi.

**Avant de commencer :**
1. Dans Colab : `Exécution > Modifier le type d'exécution > GPU (T4)`.
2. Sur Roboflow, exporter le dataset au format **YOLOv8** (zip).

Ensuite, exécuter les cellules une par une, dans l'ordre.

## 1. Vérifier le GPU et installer ultralytics

`nvidia-smi` doit afficher une carte (ex. Tesla T4). Sinon, changer le type
d'exécution (voir plus haut) : sur CPU, l'entraînement prendrait des heures.

In [ ]:
!nvidia-smi
!pip install -q ultralytics

## 2. Téléverser et préparer le dataset

Choisir le zip exporté de Roboflow. Le fichier `data.yaml` de Roboflow utilise
des chemins relatifs (`../train/images`) qui ne fonctionnent pas sur Colab :
on les remplace par des chemins absolus.

In [ ]:
import glob
import os
import zipfile

import yaml
from google.colab import files

DOSSIER_DATASET = "/content/dataset"

televerse = files.upload()
nom_zip = next(iter(televerse))
with zipfile.ZipFile(nom_zip) as z:
    z.extractall(DOSSIER_DATASET)

# Le zip peut contenir un dossier en plus : on cherche data.yaml partout
chemin_yaml = sorted(glob.glob(f"{DOSSIER_DATASET}/**/data.yaml", recursive=True))[0]
racine = os.path.dirname(chemin_yaml)

with open(chemin_yaml) as f:
    config = yaml.safe_load(f)

config["path"] = racine
config["train"] = "train/images"
config["val"] = "valid/images"
if os.path.isdir(os.path.join(racine, "test", "images")):
    config["test"] = "test/images"
else:
    config.pop("test", None)

DATA_YAML = os.path.join(racine, "data_colab.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f, allow_unicode=True)

print(f"Dataset : {racine}")
print(f"Nombre de classes : {config['nc']} (attendu : 52)")
print(config["names"])
for split in ("train", "valid", "test"):
    dossier = os.path.join(racine, split, "images")
    if os.path.isdir(dossier):
        print(f"{split} : {len(os.listdir(dossier))} images")

## 3. Entraîner le modèle

- `imgsz=640` : les coins des cartes sont petits, une image plus grande donne
  plus de précision.
- `fliplr=0.0` : pas d'effet miroir, un coin de carte inversé n'existe pas en vrai.
- `patience=20` : l'entraînement s'arrête tout seul s'il ne progresse plus.

Compter environ 1 h à 2 h sur un GPU T4. Si Colab se déconnecte, relancer la
cellule 2 puis cette cellule avec `resume=True` (voir le commentaire).

In [ ]:
from ultralytics import YOLO

modele = YOLO("yolov8n.pt")
modele.train(
    data=DATA_YAML,
    imgsz=640,
    epochs=100,
    patience=20,
    batch=16,
    fliplr=0.0,
    name="cartes",
    exist_ok=True,
)
# Reprise après déconnexion :
# YOLO("/content/runs/detect/cartes/weights/last.pt").train(resume=True)

## 4. Vérifier la précision

`mAP50` proche de 1.0 = très bon. On affiche aussi quelques prédictions sur
des images que le modèle n'a jamais vues.

In [ ]:
import glob

from IPython.display import Image, display
from ultralytics import YOLO

MEILLEUR = "/content/runs/detect/cartes/weights/best.pt"
modele = YOLO(MEILLEUR)
metriques = modele.val(data=DATA_YAML, imgsz=640)
print(f"mAP50    : {metriques.box.map50:.3f}")
print(f"mAP50-95 : {metriques.box.map:.3f}")

dossier_test = os.path.join(racine, "test", "images")
if not os.path.isdir(dossier_test):
    dossier_test = os.path.join(racine, "valid", "images")
exemples = sorted(glob.glob(f"{dossier_test}/*"))[:4]
for resultat in modele.predict(exemples, imgsz=640, conf=0.5, save=True,
                               project="/content/runs", name="exemples", exist_ok=True):
    display(Image(filename=os.path.join(resultat.save_dir, os.path.basename(resultat.path)), width=480))

## 5. Exporter pour le Raspberry Pi et télécharger

On exporte en **NCNN** (le plus rapide sur le CPU du Pi). Le zip téléchargé
contient :
- `cartes_ncnn_model/` : à copier dans `rasberry/modele/` sur le Pi ;
- `best.pt` : le modèle d'origine, à garder (pour réexporter en 416 px si le Pi est trop lent).

In [ ]:
import shutil

from ultralytics import YOLO

dossier_ncnn = YOLO(MEILLEUR).export(format="ncnn", imgsz=640)

SORTIE = "/content/modele_cartes"
shutil.rmtree(SORTIE, ignore_errors=True)
os.makedirs(SORTIE)
shutil.copytree(dossier_ncnn, os.path.join(SORTIE, "cartes_ncnn_model"))
shutil.copy(MEILLEUR, os.path.join(SORTIE, "best.pt"))

shutil.make_archive("/content/modele_cartes", "zip", SORTIE)
files.download("/content/modele_cartes.zip")